# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Di-pesh/flyinterm/blob/main/work/notebooks/w07_action_playbook.ipynb)

## 1. Ranked actions + reason codes

The output of this notebook is a ranked action queue for content professionals. The queue is not a final judgment; it is a scoring list that prioritizes pages for human review. Each recommendation should be accompanied by a reason code that explains what the model saw, such as rising risk, weak position, low freshness, or declining performance.


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import LogisticRegression

root = Path.cwd().resolve()
processed = None
for p in [root, *root.parents]:
    candidate = p / 'data' / 'processed' / 'refresh_feature_vector.csv'
    if candidate.exists():
        processed = candidate
        break
if processed is None:
    raise FileNotFoundError('Expected data/processed/refresh_feature_vector.csv. Run scripts/01_prepare_features.py first.')

df = pd.read_csv(processed)
numeric = ['search_volume', 'competition', 'cpc', 'word_count', 'char_count', 'log_impressions_90d', 'log_clicks_90d', 'log_sessions_90d', 'log_ai_sessions_90d', 'days_with_impressions', 'days_with_sessions', 'content_age_days', 'days_since_last_update', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct']
categorical = ['competition_level', 'content_type', 'main_intent', 'age_tier', 'freshness_tier', 'word_count_tier', 'impression_tier', 'position_tier']
features = [c for c in numeric + categorical if c in df.columns]
X = df[features]
y = df['is_declining_label'].astype(int)
pre = ColumnTransformer([
    ('num', SimpleImputer(strategy='median'), [c for c in numeric if c in X.columns]),
    ('cat', Pipeline([('imputer', SimpleImputer(strategy='most_frequent')), ('ohe', OneHotEncoder(handle_unknown='ignore'))]), [c for c in categorical if c in X.columns]),
])
model = Pipeline([('pre', pre), ('lr', LogisticRegression(max_iter=2000, class_weight='balanced'))])
model.fit(X, y)
df = df.copy()
df['risk_score'] = model.predict_proba(X)[:, 1]
df['reason_code'] = np.select(
    [
        df['avg_position'].replace(0, np.nan).gt(df['avg_position'].replace(0, np.nan).median()),
        df['content_age_days'].gt(df['content_age_days'].median()),
        df['ai_traffic_pct'].gt(df['ai_traffic_pct'].median()),
    ],
    ['LOW_POSITION', 'OLD_CONTENT', 'AI_TRAFFIC_RISK'],
    default='GENERAL_RISK'
)
queue = df[['content_id', 'client_id', 'risk_score', 'reason_code', 'avg_position', 'content_age_days', 'ai_traffic_pct']].sort_values('risk_score', ascending=False).head(20)
print(queue.to_string(index=False))
out_dir = root / 'work' / 'outputs'
out_dir.mkdir(parents=True, exist_ok=True)
queue.to_csv(out_dir / 'action_queue.csv', index=False)
print(f'
Saved queue to {out_dir / "action_queue.csv"}')


## 2. Intended use and limits

This queue is designed to support a human review process. It tells a content team which pages warrant attention first but it is not a final automated action trigger. It is useful as a prioritization signal, not as a proof that any single page must be updated or rewritten.

The limits are important: the model is observational, based on historical patterns, and may fail on novel content types or unusual client behavior. Human review should decide the final action.


In [ ]:
print('Use: ranking and review support. Limit: no automated decision without human review.')


## 3. Human review + the no-go list

Before acting, a human should check whether the page is strategically important, whether a refresh would be brand-safe, whether there are seasonal or campaign effects in the traffic, and whether the model is over-weighting a stale signal. The no-go list should include: pages with legal or policy risk, pages tied to an active campaign, pages with severe data sparsity, and pages whose top-risk drivers are not explainable.


In [ ]:
no_go = [
    'Any page with legal, policy, or brand-risk concerns',
    'Any page with a live campaign or revenue-critical schedule',
    'Any page that is edge-case sparse or missing key content metadata',
    'Any page where the reason code is not explainable to a content editor',
]
for item in no_go:
    print('-', item)


## 4. Monitoring / retrain triggers

The queue should be monitored for drift. If the distribution of risk scores shifts materially, if page refresh outcomes stop matching the expected pattern, or if the top 20 queue changes after a product or seasonal event, the model should be retrained or re-audited.


In [ ]:
triggers = [
    'Risk-score distribution shifts materially over two or more weeks',
    'Human review finds repeated false positives in a major content type',
    'Traffic mix changes because of seasonality, campaign launches, or major product changes',
    'Reason codes become unstable or less explainable across clients',
]
for t in triggers:
    print('-', t)


## 5. Exports for the paper

The final queue and the key charts should be exported to `work/outputs/` so the capstone or narrative can cite the real recommendation list and the observed supporting metrics.


In [ ]:
from pathlib import Path
out_dir = Path('work/outputs')
out_dir.mkdir(parents=True, exist_ok=True)
(out_dir / 'playbook_notes.txt').write_text('Action playbook exported for human review and downstream reporting.\n')
print(f'Prepared exports in {out_dir}')


## Self-check

- [x] The notebook produces a ranked queue and human-readable reason codes.
- [x] It states intended use and explicit limits.
- [x] It includes a human review checklist and a no-go list.
- [x] It documents monitoring and export steps for downstream reporting.
